# 02 - Build the encoder embedding dataset 🧬

The default profile uses NVIDIA ESM-2 and IBM MoLFormer ONNX encoders. Set `ENCODER_PROFILE = 'legacy'` to use ProLLaMA and Mol-LLaMA exports.

Rows are stored in CSV files. High-dimensional feature matrices are stored in compressed NPZ files beside the CSV files.

In [ ]:
WK_DIR = "/content/"

In [ ]:
%cd {WK_DIR}
!test -d Protein-Ligand-Affinity-Prediction-Using-LLM || git clone https://github.com/karthikeyanr103/Protein-Ligand-Affinity-Prediction-Using-LLM.git
%cd {WK_DIR}/Protein-Ligand-Affinity-Prediction-Using-LLM
import shutil
import subprocess

GPU_AVAILABLE = bool(shutil.which("nvidia-smi")) and subprocess.run(
    ["nvidia-smi"], capture_output=True, check=False
).returncode == 0
RUNTIME_EXTRA = "gpu-runtime" if GPU_AVAILABLE else "runtime"
print(f"Installing ONNX {RUNTIME_EXTRA} dependencies")
%pip install -e ".[{RUNTIME_EXTRA}]" "huggingface-hub>=0.30"

In [ ]:
import os, subprocess
import kagglehub
from google.colab import userdata
from huggingface_hub import HfApi, login, snapshot_download
from pathlib import Path
os.sys.path.insert(0,f'{WK_DIR}/Protein-Ligand-Affinity-Prediction-Using-LLM/src')

HF_USER = userdata.get('HF_USER')
token = userdata.get('HF_TOKEN')
login(token=token)
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
ENCODER_PROFILE = 'lightweight'  # lightweight or legacy
PROTEIN_ENCODER = 'esm2'
MOLECULE_ENCODER = 'molformer'
PROTEIN_MODEL_ID = 'facebook/esm2_t12_35M_UR50D'
MOLECULE_MODEL_ID = 'ibm-research/MoLFormer-XL-both-10pct'
PROTEIN_REPO = f'{HF_USER}/esm2-affinity-onnx'
MOLECULE_REPO = f'{HF_USER}/molformer-affinity-onnx'
DATASET_REPO = f'{HF_USER}/protein-compound-affinity-esm2-molformer'
if ENCODER_PROFILE == 'legacy':
    PROTEIN_ENCODER = 'prollama'
    MOLECULE_ENCODER = 'mol_llama'
    PROTEIN_MODEL_ID = 'GreatCaptainNemo/ProLLaMA'
    MOLECULE_MODEL_ID = 'DongkiKim/Mol-Llama-3.1-8B-Instruct'
    PROTEIN_REPO = f'{HF_USER}/prollama-affinity-onnx'
    MOLECULE_REPO = f'{HF_USER}/mol-llama-affinity-onnx'
    DATASET_REPO = f'{HF_USER}/protein-compound-affinity-embeddings'
ROOT = Path(f'{WK_DIR}/protein_affinity')
RAW_DATA = ROOT / 'data/train.csv'
CACHE = ROOT / 'embedding_cache'
DATASET_OUT = ROOT / 'embedding_dataset'
CACHE.mkdir(parents=True, exist_ok=True)
RAW_DATA.parent.mkdir(parents=True, exist_ok=True)
DATASET_OUT.mkdir(parents=True, exist_ok=True)
kagglehub.competition_download('protein-compound-affinity',output_dir = ROOT/'data',force_download=True)
assert RAW_DATA.exists(), f'Missing: {RAW_DATA}'
PROTEIN_ONNX = Path(snapshot_download(PROTEIN_REPO, token=token))
MOLECULE_ONNX = Path(snapshot_download(MOLECULE_REPO, token=token))

## Inspect the source data

In [ ]:
from affinity.data import load_dataset, profile_dataset
frame = load_dataset(RAW_DATA)
profile_dataset(frame)

## Extract ONNX embeddings

Protein embeddings are stored in one file. Molecule embeddings are sharded so extraction can resume after a disconnect.

In [ ]:
subprocess.run([
    'affinity-extract-onnx',
    '--data', str(RAW_DATA),
    '--protein-onnx', str(PROTEIN_ONNX),
    '--molecule-onnx', str(MOLECULE_ONNX),
    '--protein-encoder', PROTEIN_ENCODER,
    '--molecule-encoder', MOLECULE_ENCODER,
    '--protein-model-id', PROTEIN_MODEL_ID,
    '--molecule-model-id', MOLECULE_MODEL_ID,
    '--protein-output', str(CACHE / f'{PROTEIN_ENCODER}_embeddings.npz'),
    '--molecule-output', str(CACHE / MOLECULE_ENCODER),
    '--protein-max-length', '1024' if PROTEIN_ENCODER == 'esm2' else '512',
    '--molecule-max-length', '202',
    '--protein-batch-size', '16' if PROTEIN_ENCODER == 'esm2' else '1',
    '--molecule-batch-size', '32' if MOLECULE_ENCODER == 'molformer' else '1',
    '--molecule-shard-size', '1000',
    '--device', 'auto',
], check=True)

## Data Preparation

In [ ]:
subprocess.run([
    'affinity-prepare-dataset',
    '--data', str(RAW_DATA),
    '--protein-embeddings', str(CACHE / f'{PROTEIN_ENCODER}_embeddings.npz'),
    '--molecule-embeddings', str(CACHE / MOLECULE_ENCODER),
    '--output', str(DATASET_OUT),
    '--split-strategy', 'cold_protein',
    '--seed', '42',
], check=True)
print(sorted(path.name for path in DATASET_OUT.iterdir()))

In [ ]:
import pandas as pd
import numpy as np
import json
for split in ['train', 'validation', 'test']:
    rows = pd.read_csv(DATASET_OUT / f'{split}.csv')
    features = np.load(DATASET_OUT / f'{split}_features.npz')['features']
    print(split, 'rows=', len(rows), 'features=', features.shape)
print(json.loads((DATASET_OUT / 'dataset_metadata.json').read_text()))

## Upload the embedding dataset

In [ ]:
UPLOAD = False
if UPLOAD:
    api = HfApi(token=token)
    api.create_repo(DATASET_REPO, repo_type='dataset', exist_ok=True)
    api.upload_large_folder(
        repo_id=DATASET_REPO,
        repo_type='dataset',
        folder_path=str(DATASET_OUT),
    )
    print('Uploaded:', DATASET_REPO)